# Sieć Hopfielda jako pamięć skojarzeniowa: demo

Interaktywna prezentacja do projektu 2. Wybierz zbiór, regułę uczenia, tryb symulacji (synchroniczny / asynchroniczny) i poziom szumu, a potem przewijaj kolejne kroki zbiegania sieci suwakiem `krok`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, clear_output, display

import experiments as E
from hopfield import Hopfield, LowRankHopfield
from hopfield_datasets import DATA_DIR, DATASETS, load_dataset, load_images

## 1. Odtwarzanie zaszumionego wzorca krok po kroku

In [ ]:
ds = w.Dropdown(options=DATASETS, value="animals-14x9", description="zbiór")
rule = w.Dropdown(options=["hebb", "oja", "delta", "projection"], description="reguła")
mode = w.ToggleButtons(options=[("synchroniczny", "sync"), ("asynchroniczny", "async")])
noise = w.FloatSlider(value=0.15, min=0, max=0.5, step=0.05, description="szum")
pattern = w.IntSlider(value=0, min=0, max=5, description="wzorzec")
seed = w.IntText(value=E.SEED, description="ziarno")
step = w.IntSlider(value=0, min=0, max=0, description="krok")
out = w.Output()
state = {}


def run(_=None):
    d = load_dataset(ds.value)
    pattern.max = d.n_patterns - 1
    net = E.trained_network(d, rule.value)
    x = net.corrupt(d.patterns[pattern.value], noise.value, np.random.default_rng(seed.value))
    res = net.run(x, mode=mode.value, seed=seed.value)
    state.update(d=d, net=net, res=res, energy=[net.energy(h) for h in res.history])
    step.max = res.steps
    step.value = res.steps
    draw()


def draw(_=None):
    if not state:
        return
    d, net, res, energy = state["d"], state["net"], state["res"], state["energy"]
    k = min(step.value, res.steps)
    with out:
        clear_output(wait=True)
        _, axes = plt.subplots(1, 4, figsize=(13, 3), gridspec_kw={"width_ratios": [1, 1, 1, 1.4]})
        net.plot_pattern(d.patterns[pattern.value], ax=axes[0], title="oryginał")
        net.plot_pattern(res.history[0], ax=axes[1], title=f"wejście ({noise.value:.0%} szumu)")
        net.plot_pattern(res.history[k], ax=axes[2], title=f"krok {k} / {res.steps}")
        axes[3].plot(energy, "o-")
        axes[3].plot(k, energy[k], "ro")
        axes[3].set_xlabel("krok")
        axes[3].set_title("energia E(s)")
        plt.show()
        stable = net.stable_patterns()
        print(f"stabilne wzorce uczące: {stable.sum()} / {len(stable)}  {np.flatnonzero(stable)}")
        outcome = E.classify(net, res, pattern.value)
        print(f"wynik: {outcome}, zbieżność={res.converged}, cykl={res.cycle_length}")


for widget in (ds, rule, mode, noise, pattern, seed):
    widget.observe(run, names="value")
step.observe(draw, names="value")
display(w.VBox([w.HBox([ds, rule, pattern]), w.HBox([mode, noise, seed]), step, out]))
run()

## 2. Dowolna liczba neuronów

Rozmiar sieci to `rows * cols` neuronów; tu z losowymi wzorcami, żeby pokazać wpływ obciążenia $P/N$ (pojemność reguły Hebba $\approx 0.138 N$).

In [ ]:
@w.interact(rows=(3, 40), cols=(3, 40), P=(1, 60), rule=["hebb", "projection", "oja"])
def random_patterns(rows=10, cols=10, P=8, rule="hebb"):
    rng = np.random.default_rng(E.SEED)
    X = rng.choice([-1.0, 1.0], (P, rows * cols))
    net = Hopfield((rows, cols), X)
    net.train(rule)
    print(
        f"N = {net.size}, P/N = {P / net.size:.3f}, stabilne: {net.stable_patterns().sum()} / {P}"
    )

## 3. Synchronicznie vs asynchronicznie: animacje GIF

In [ ]:
d = load_dataset("OCRA-12x30-cut")
net = E.trained_network(d, "projection")
x = net.corrupt(d.patterns[5], 0.4, np.random.default_rng(E.SEED))
for m in ("sync", "async"):
    # asynchronous sweeps are split into 4 snapshots to show the gradual updates
    res = net.run(x, mode=m, seed=E.SEED, frames_per_sweep=4)
    net.save_history_gif(res.history, f"convergence_{m}.gif", fps=2)
    print(m, "kroki:", res.steps)
    display(Image(filename=f"convergence_{m}.gif"))

## 4. Oscylacja w trybie synchronicznym

Sieć 6x6 zapamiętuje jeden wzorzec $x$, a wejście $s$ jest do niego ortogonalne ($x \cdot s = 0$). Wtedy $h = -s/N$, więc krok synchroniczny odwraca wszystkie neurony: $s \to -s \to s$. Tryb asynchroniczny zbiega (energia maleje).

In [ ]:
o = E.oscillation_example()
print("sync: cykl długości", o["sync"].cycle_length, "energia", o["sync_energy"])
print("async: zbieżność", o["async"].converged, "energia", np.round(o["async_energy"], 2))
o["net"].plot_history([*o["sync"].history, o["sync"].state, *o["async"].history], cols=6)

## 5. Duże bitmapy (200x300, N = 60 000): koty

Macierz wag 60 000 x 60 000 nie mieści się w pamięci, więc `LowRankHopfield` liczy pole lokalne przez nakładania $m = Xs$ (pamięć $O(PN)$). Najpierw `uv run python scripts/fetch_cats.py`.

In [ ]:
cats = load_images(sorted((DATA_DIR / "cats").iterdir()), (200, 300))
big = LowRankHopfield(cats.shape, cats.patterns, rule="hebb")
print("stabilne:", big.stable_patterns().sum(), "/", cats.n_patterns)


@w.interact(cat=(0, cats.n_patterns - 1), noise=(0.0, 0.5, 0.05), rule=["hebb", "projection"])
def cat_demo(cat=0, noise=0.35, rule="hebb"):
    net = LowRankHopfield(cats.shape, cats.patterns, rule=rule)
    x = net.corrupt(cats.patterns[cat], noise, np.random.default_rng(E.SEED))
    res = net.run(x, mode="sync", max_iter=20)
    net.plot_history(res.history, cols=min(4, len(res.history)))